In [3]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)
import os

path = "/content/drive/MyDrive/Tomato"
print(os.listdir(path))

Mounted at /content/drive
['Tomato___Tomato_mosaic_virus', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Bacterial_spot', 'Tomato___healthy', 'Tomato___Early_blight', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot', 'Tomato___Septoria_leaf_spot', 'Tomato___Leaf_Mold', 'Tomato___Late_blight']


In [4]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torchvision.models import googlenet, GoogLeNet_Weights
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
import numpy as np
import random

# ================= CONFIG =================
# DATA_DIR is defined in a previous cell
SAVE_DIR   = "saved_models"
DATA_DIR   = path
BATCH_SIZE = 32
EPOCHS     = 10
LR         = 0.0003
FOLDS      = 5
SEED       = 42

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ================= TRANSFORM =================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)

# ================= TOMATO FILTER =================
tomato_class_indices = {
    idx: name
    for name, idx in full_dataset.class_to_idx.items()
    if "tomato" in name.lower()
}

print(f"\nTomato classes found ({len(tomato_class_indices)}):")
for idx, name in sorted(tomato_class_indices.items()):
    print(f"  [{idx}] {name}")

# Re-map old class indices -> new contiguous indices (0, 1, 2, ...)
old_to_new  = {old: new for new, old in enumerate(sorted(tomato_class_indices.keys()))}
num_classes = len(old_to_new)

# Collect ALL sample positions that belong to tomato classes
tomato_positions = [
    i for i, (_, lbl) in enumerate(full_dataset.samples)
    if lbl in tomato_class_indices
]
tomato_labels    = np.array([old_to_new[full_dataset.samples[i][1]] for i in tomato_positions])
tomato_positions = np.array(tomato_positions)

print(f"\nTotal tomato samples: {len(tomato_positions)}")
print("Per-class counts:")
for new_lbl in range(num_classes):
    class_name = tomato_class_indices[sorted(tomato_class_indices.keys())[new_lbl]]
    count = (tomato_labels == new_lbl).sum()
    print(f"  [{new_lbl}] {class_name}: {count}")

# ================= REMAPPED SUBSET WRAPPER =================
class RemappedSubset(torch.utils.data.Dataset):
    """Wraps ImageFolder, applies a label remapping, and exposes a subset by positions."""
    def __init__(self, dataset, positions, label_map):
        self.dataset   = dataset
        self.positions = positions
        self.label_map = label_map

    def __len__(self):
        return len(self.positions)

    def __getitem__(self, i):
        img, lbl = self.dataset[self.positions[i]]
        return img, self.label_map[lbl]

remapped_dataset = RemappedSubset(full_dataset, tomato_positions, old_to_new)

# ================= 2-WAY SPLIT  (80 train+val / 20 test) =================
# FIX 1: Removed the unused fixed val set. StratifiedKFold now handles all
# train/val splitting internally over the full 80% pool. Only the 20% test
# set is carved out here, and it is evaluated ONCE per fold — after the
# best checkpoint has already been chosen by val F1.
train_pos, test_pos, train_lbl, _ = train_test_split(
    np.arange(len(tomato_positions)),
    tomato_labels,
    test_size=0.20,
    stratify=tomato_labels,
    random_state=SEED
)

print(f"\nSplit sizes  ->  Train+Val pool: {len(train_pos)} | Test (held-out): {len(test_pos)}")

# ================= COORDINATE ATTENTION BLOCK =================
class CoordinateAttention(nn.Module):
    """
    Coordinate Attention (Hou et al., CVPR 2021).
    Encodes spatial information along H and W axes separately,
    then fuses them to produce channel-wise + positional attention.
    """
    def __init__(self, in_channels, reduction=32):
        super().__init__()
        mid = max(8, in_channels // reduction)

        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))

        self.conv1  = nn.Conv2d(in_channels, mid, kernel_size=1, bias=False)
        self.bn1    = nn.BatchNorm2d(mid)
        self.act    = nn.Hardswish()

        self.conv_h = nn.Conv2d(mid, in_channels, kernel_size=1, bias=False)
        self.conv_w = nn.Conv2d(mid, in_channels, kernel_size=1, bias=False)

    def forward(self, x):
        B, C, H, W = x.shape

        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)

        y   = torch.cat([x_h, x_w], dim=2)
        y   = self.act(self.bn1(self.conv1(y)))

        x_h_, x_w_ = torch.split(y, [H, W], dim=2)
        x_w_ = x_w_.permute(0, 1, 3, 2)

        a_h = torch.sigmoid(self.conv_h(x_h_))
        a_w = torch.sigmoid(self.conv_w(x_w_))

        return x * a_h * a_w

# ================= SE BLOCK =================
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y

# ================= MODEL =================
class ModifiedGoogLeNet(nn.Module):
    """
    GoogLeNet backbone with:
      - SE blocks  (channel recalibration) + residual skip after each phase
      - Coordinate Attention  (spatial recalibration) after each SE+skip block
      - Dropout before final classifier
    """
    def __init__(self, num_classes):
        super().__init__()

        base = googlenet(weights=GoogLeNet_Weights.DEFAULT)
        base.aux_logits = False
        base.aux1       = None
        base.aux2       = None

        self.phase1 = nn.Sequential(
            base.conv1, base.maxpool1,
            base.conv2, base.conv3, base.maxpool2,
        )
        self.phase2 = nn.Sequential(
            base.inception3a, base.inception3b, base.maxpool3,
        )
        self.phase3 = nn.Sequential(
            base.inception4a, base.inception4b, base.inception4c,
            base.inception4d, base.inception4e, base.maxpool4,
        )
        self.phase4 = nn.Sequential(
            base.inception5a, base.inception5b,
        )

        self.se12 = SEBlock(192)
        self.se23 = SEBlock(480)
        self.se34 = SEBlock(832)

        self.ca12 = CoordinateAttention(192)
        self.ca23 = CoordinateAttention(480)
        self.ca34 = CoordinateAttention(832)
        self.ca4  = CoordinateAttention(1024)

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.drop = nn.Dropout(p=0.4)
        self.fc   = nn.Linear(1024, num_classes)

    def forward(self, x):
        x = self.phase1(x)
        x = self.se12(x) + x
        x = self.ca12(x)

        x = self.phase2(x)
        x = self.se23(x) + x
        x = self.ca23(x)

        x = self.phase3(x)
        x = self.se34(x) + x
        x = self.ca34(x)

        x = self.phase4(x)
        x = self.ca4(x)

        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.drop(x)
        x = self.fc(x)
        return x

# ================= METRICS HELPER =================
def compute_metrics(labels, preds, probs):
    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, average='macro', zero_division=0)
    rec  = recall_score(labels,   preds,  average='macro', zero_division=0)
    f1   = f1_score(labels,       preds,  average='macro', zero_division=0)
    try:
        auc = roc_auc_score(labels, probs, multi_class='ovr')
    except Exception:
        auc = float('nan')
    return acc, prec, rec, f1, auc

def evaluate(model, loader):
    """Run inference; returns (preds, labels, probs)."""
    model.eval()
    preds_all, labels_all, probs_all = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs  = model(images)
            probs    = F.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            preds_all.extend(predicted.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
            probs_all.extend(probs.cpu().numpy())
    return np.array(preds_all), np.array(labels_all), np.array(probs_all)

def print_metrics(split, acc, prec, rec, f1, auc=None):
    auc_str = f"{auc:.4f}" if auc is not None and not np.isnan(auc) else "  N/A  "
    print(f"  {split:<6}  Acc: {acc:.4f}  Prec: {prec:.4f}  Rec: {rec:.4f}  F1: {f1:.4f}  AUC: {auc_str}")

# ================= K-FOLD TRAINING =================
# StratifiedKFold now operates on the full 80% train pool.
# Each fold: ~64% train, ~16% val. Test set is fully isolated.
skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

fold_results = {split: {m: [] for m in ['acc', 'prec', 'rec', 'f1', 'auc']}
                for split in ['train', 'val', 'test']}

global_best_val_f1  = 0.0
global_best_fold    = -1
global_best_ckpt    = None
global_best_metrics = None

# FIX 2: Build test loader once outside the fold loop.
# num_workers=0 avoids multiprocessing issues on Windows / notebooks.
test_subset = Subset(remapped_dataset, test_pos)
test_loader = DataLoader(
    test_subset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=True
)

for fold, (tr_idx, val_idx) in enumerate(skf.split(train_pos, train_lbl)):
    print(f"\n{'='*65}")
    print(f"  FOLD {fold+1}/{FOLDS}")
    print(f"{'='*65}")

    fold_train_subset = Subset(remapped_dataset, train_pos[tr_idx])
    fold_val_subset   = Subset(remapped_dataset, train_pos[val_idx])

    train_loader = DataLoader(
        fold_train_subset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=0, pin_memory=True
    )
    val_loader = DataLoader(
        fold_val_subset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=0, pin_memory=True
    )

    print(f"  Fold train samples : {len(fold_train_subset)}")
    print(f"  Fold val  samples  : {len(fold_val_subset)}")
    print(f"  Test      samples  : {len(test_subset)}  (held out — not seen during training)")

    model     = ModifiedGoogLeNet(num_classes).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_val_f1    = 0.0
    best_train_m   = None
    best_val_m     = None
    best_fold_ckpt = None

    for epoch in range(EPOCHS):
        # -------- TRAIN --------
        model.train()
        train_preds_ep, train_labels_ep, train_probs_ep = [], [], []

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            probs = F.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            train_preds_ep.extend(predicted.cpu().numpy())
            train_labels_ep.extend(labels.cpu().numpy())
            train_probs_ep.extend(probs.detach().cpu().numpy())

        scheduler.step()

        train_preds_ep  = np.array(train_preds_ep)
        train_labels_ep = np.array(train_labels_ep)
        train_probs_ep  = np.array(train_probs_ep)

        # -------- VAL ONLY — test set never touched here --------
        # FIX 3: Test set is NOT evaluated each epoch. Only train and
        # fold-val metrics are computed. Checkpoint selection uses val F1
        # exclusively, so the test set has zero influence on model selection.
        val_preds, val_labels, val_probs = evaluate(model, val_loader)

        train_m = compute_metrics(train_labels_ep, train_preds_ep, train_probs_ep)
        val_m   = compute_metrics(val_labels, val_preds, val_probs)

        print(f"\n  Epoch {epoch+1}/{EPOCHS}  (lr={scheduler.get_last_lr()[0]:.6f})")
        print(f"  {'-'*55}")
        print(f"  {'Split':<6}  {'Acc':>6}  {'Prec':>6}  {'Rec':>6}  {'F1':>6}  {'AUC':>7}")
        print(f"  {'-'*55}")
        print_metrics("Train", *train_m)
        print_metrics("Val",   *val_m)
        print(f"  {'-'*55}")

        val_f1 = val_m[3]
        if val_f1 > best_val_f1:
            best_val_f1    = val_f1
            best_train_m   = train_m
            best_val_m     = val_m
            best_fold_ckpt = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f"  ✔ New best val F1: {best_val_f1:.4f}  — checkpoint saved")

    # -------- Evaluate test set ONCE using the best checkpoint --------
    # Test evaluation happens exactly once per fold, after the best epoch
    # has been selected purely by val F1. Clean, unbiased test estimate.
    print(f"\n  Loading best checkpoint (val F1={best_val_f1:.4f}) for final test evaluation...")
    model.load_state_dict(best_fold_ckpt)
    test_preds, test_labels, test_probs = evaluate(model, test_loader)
    test_m = compute_metrics(test_labels, test_preds, test_probs)

    # ---- Fold summary ----
    print(f"\n  ★ Fold {fold+1} — best-epoch metrics:")
    print(f"  {'-'*55}")
    print_metrics("Train", *best_train_m)
    print_metrics("Val",   *best_val_m)
    print_metrics("Test",  *test_m)
    print(f"  {'-'*55}")

    for key, val in zip(['acc', 'prec', 'rec', 'f1', 'auc'], best_train_m):
        fold_results['train'][key].append(val)
    for key, val in zip(['acc', 'prec', 'rec', 'f1', 'auc'], best_val_m):
        fold_results['val'][key].append(val)
    for key, val in zip(['acc', 'prec', 'rec', 'f1', 'auc'], test_m):
        fold_results['test'][key].append(val)

    # ---- Save per-fold best checkpoint ----
    fold_ckpt_path = os.path.join(SAVE_DIR, f"fold_{fold+1}_best.pth")
    torch.save({
        'fold'          : fold + 1,
        'num_classes'   : num_classes,
        'class_names'   : [tomato_class_indices[sorted(tomato_class_indices.keys())[i]]
                           for i in range(num_classes)],
        'old_to_new'    : old_to_new,
        'val_f1'        : best_val_f1,
        'train_metrics' : dict(zip(['acc','prec','rec','f1','auc'], best_train_m)),
        'val_metrics'   : dict(zip(['acc','prec','rec','f1','auc'], best_val_m)),
        'test_metrics'  : dict(zip(['acc','prec','rec','f1','auc'], test_m)),
        'model_state'   : best_fold_ckpt,
    }, fold_ckpt_path)
    print(f"\n  ✔ Fold {fold+1} checkpoint saved -> {fold_ckpt_path}")

    if best_val_f1 > global_best_val_f1:
        global_best_val_f1  = best_val_f1
        global_best_fold    = fold + 1
        global_best_ckpt    = best_fold_ckpt
        global_best_metrics = (best_train_m, best_val_m, test_m)

# ================= FINAL RESULTS =================
print(f"\n{'='*65}")
print("  FINAL CROSS-VALIDATION RESULTS  (mean +/- std across folds)")
print(f"{'='*65}")
print(f"  {'Metric':<9}  {'Train':^20}  {'Val':^20}  {'Test':^20}")
print(f"  {'-'*65}")
for metric in ['acc', 'prec', 'rec', 'f1', 'auc']:
    row = f"  {metric.upper():<9}"
    for split in ['train', 'val', 'test']:
        arr = np.array(fold_results[split][metric])
        row += f"  {arr.mean():.4f} +/- {arr.std():.4f}  "
    print(row)
print(f"{'='*65}\n")

# ================= SAVE FINAL (GLOBALLY BEST) MODEL =================
if global_best_ckpt is not None:
    final_model_path = os.path.join(SAVE_DIR, "best_model_final.pth")
    class_names      = [tomato_class_indices[sorted(tomato_class_indices.keys())[i]]
                        for i in range(num_classes)]
    tr_m, vl_m, ts_m = global_best_metrics

    torch.save({
        'fold'          : global_best_fold,
        'num_classes'   : num_classes,
        'class_names'   : class_names,
        'old_to_new'    : old_to_new,
        'val_f1'        : global_best_val_f1,
        'train_metrics' : dict(zip(['acc','prec','rec','f1','auc'], tr_m)),
        'val_metrics'   : dict(zip(['acc','prec','rec','f1','auc'], vl_m)),
        'test_metrics'  : dict(zip(['acc','prec','rec','f1','auc'], ts_m)),
        'model_state'   : global_best_ckpt,
    }, final_model_path)

    meta_path = os.path.join(SAVE_DIR, "class_names.json")
    with open(meta_path, "w") as f:
        json.dump(class_names, f, indent=2)

    print(f"\n{'='*65}")
    print(f"  ✔ FINAL MODEL saved  ->  {final_model_path}")
    print(f"  ✔ Class names saved  ->  {meta_path}")
    print(f"  Best fold : {global_best_fold}  |  Val F1 : {global_best_val_f1:.4f}")
    print(f"{'='*65}\n")

Using device: cuda

Tomato classes found (10):
  [0] Tomato___Bacterial_spot
  [1] Tomato___Early_blight
  [2] Tomato___Late_blight
  [3] Tomato___Leaf_Mold
  [4] Tomato___Septoria_leaf_spot
  [5] Tomato___Spider_mites Two-spotted_spider_mite
  [6] Tomato___Target_Spot
  [7] Tomato___Tomato_Yellow_Leaf_Curl_Virus
  [8] Tomato___Tomato_mosaic_virus
  [9] Tomato___healthy

Total tomato samples: 18160
Per-class counts:
  [0] Tomato___Bacterial_spot: 2127
  [1] Tomato___Early_blight: 1000
  [2] Tomato___Late_blight: 1909
  [3] Tomato___Leaf_Mold: 952
  [4] Tomato___Septoria_leaf_spot: 1771
  [5] Tomato___Spider_mites Two-spotted_spider_mite: 1676
  [6] Tomato___Target_Spot: 1404
  [7] Tomato___Tomato_Yellow_Leaf_Curl_Virus: 5357
  [8] Tomato___Tomato_mosaic_virus: 373
  [9] Tomato___healthy: 1591

Split sizes  ->  Train+Val pool: 14528 | Test (held-out): 3632

  FOLD 1/5
  Fold train samples : 11622
  Fold val  samples  : 2906
  Test      samples  : 3632  (held out — not seen during traini

In [5]:
!zip -r my_model.zip /content/saved_models

  adding: content/saved_models/ (stored 0%)
  adding: content/saved_models/class_names.json (deflated 54%)
  adding: content/saved_models/fold_4_best.pth (deflated 7%)
  adding: content/saved_models/fold_3_best.pth (deflated 7%)
  adding: content/saved_models/fold_5_best.pth (deflated 7%)
  adding: content/saved_models/fold_2_best.pth (deflated 7%)
  adding: content/saved_models/fold_1_best.pth (deflated 7%)
  adding: content/saved_models/best_model_final.pth (deflated 7%)


In [6]:
from google.colab import files
files.download('my_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>